# 31 - Blending models with different encoders (experiment 15)

Blending had the highest priority among the options after experiment 14. Submissions 22-24 and 27-29 use two kinds of model, two feature sets and three team
encoders: their errors may differ enough for an average to help. This notebook also answers an open question: on 15 folds, is soft
voting still better than LightGBM alone?

Two blends, with weights fixed in advance and never tuned:

- **Boosting blend:** plain mean of the probabilities of the six models of submissions 22, 23, 24, 27, 28 and 29. All six, so that
  nothing is chosen by its public score;
- **Blend + LogReg:** 75% boosting blend, 25% logistic regression (tuned C of experiment 04) with all features and the R1 + R3
  encoder. LogReg is weaker alone (0.6843 against 0.6888 for LightGBM), but as a linear model it may make different errors. Whether
  that helps has not been measured: this is the test.

**Decision rule (fixed before running):** both comparisons are paired and must pass on seeds 42 / 7 / 123 **and** on the new
seeds 1 / 2 / 3, 15 folds each.
- Boosting blend against soft voting of submission 28: at least **+0.15 points** on both sets of seeds.
- Blend + LogReg against the boosting blend: at least **+0.15 points** on both sets of seeds; otherwise LogReg stays out and is
  reported as a negative result.

+0.15 points is below the 0.5 usually required, because equal weights choose nothing and an average adds no selection risk. It
remains a small gain, close to the noise of the measure.

## 0. Setup

In [1]:
import os
import sys
import json
import pickle
import time
import warnings
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import VotingClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import robust_columns, team_key_features
from src.models import get_model_specs, with_params
from src.team_strength import KEY_P1, KEY_P2, with_team_strength
from src.validation import REPEAT_SEEDS, fold_accuracy, oof_proba

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
PROC = ROOT / "data" / "processed"
OUT = PROC / "models"
OUT.mkdir(parents=True, exist_ok=True)

team, states = [pd.read_pickle(PROC / f"{name}_train.pkl") for name in ("team", "states")]
ref = pd.read_pickle(PROC / "features_train.pkl")             # v1 features (notebook 03)
with open(PROC / "feature_groups.json") as f:
    groups = json.load(f)
y = ref.pop("player_won")
NUM = [c for c in ref.columns if c not in groups["species_cat"]]
ROBUST = robust_columns(NUM)
KEYS = [KEY_P1, KEY_P2]
X = pd.concat([ref[NUM], team_key_features(team, states).reindex(ref.index)], axis=1)
assert X[KEYS].notna().all().all()
print(f"battles: {len(X):,} | all features: {len(NUM)} | robust features: {len(ROBUST)}")

battles: 10,000 | all features: 155 | robust features: 145


## 1. Models

In [2]:
# Tuned in notebooks/04_models.ipynb (experiment 04), the same parameters as the submissions
PARAMS = {
    "xgboost": {
        "model__colsample_bytree": 0.6856273981493477, "model__gamma": 0.010493314645539311,
        "model__learning_rate": 0.012145928554953287, "model__max_depth": 6,
        "model__min_child_weight": 1.3386261584543908, "model__n_estimators": 720,
        "model__reg_lambda": 3.5416296712651336, "model__subsample": 0.5993578407670862,
    },
    "lightgbm": {
        "model__colsample_bytree": 0.6286197472598067, "model__learning_rate": 0.006552992991573526,
        "model__min_child_samples": 106, "model__n_estimators": 1424, "model__num_leaves": 12,
        "model__reg_alpha": 0.6070874110546393, "model__reg_lambda": 2.7129089558470034,
        "model__subsample": 0.6629794526009424,
    },
    "catboost": {
        "model__bagging_temperature": 0.15071754396542947, "model__depth": 5, "model__iterations": 763,
        "model__l2_leaf_reg": 1.14876138417489, "model__learning_rate": 0.03898458749305194,
        "model__random_strength": 1.3551287236845648,
    },
    "gradient_boosting": {
        "model__learning_rate": 0.04760153910846178, "model__max_depth": 3, "model__max_features": None,
        "model__min_samples_leaf": 91, "model__n_estimators": 466, "model__subsample": 0.9314950036607718,
    },
}
LOGREG_PARAMS = {"model__C": 0.006690421166498805}

specs = get_model_specs()
LGBM = with_params(specs["lightgbm"], PARAMS["lightgbm"])
LOGREG = with_params(specs["logreg"], LOGREG_PARAMS)
VOTING = VotingClassifier([(n, with_params(specs[n], PARAMS[n])) for n in PARAMS], voting="soft", n_jobs=1)

ENCODER_22 = {"train_unseen": 0.3}                                   # also submission 24
ENCODER_23 = {"counts": False, "train_unseen": 0.5}
R1_R3 = {"train_unseen": 0.3, "pairs": True, "per_side": True}       # submissions 27-29

ALL = NUM + KEYS
ROB = ROBUST + KEYS
MEMBERS = {
    "22 voting": (with_team_strength(VOTING, **ENCODER_22), ALL),
    "23 voting robust": (with_team_strength(VOTING, **ENCODER_23), ROB),
    "24 LightGBM": (with_team_strength(LGBM, **ENCODER_22), ALL),
    "27 LightGBM R1+R3": (with_team_strength(LGBM, **R1_R3), ALL),
    "28 voting R1+R3": (with_team_strength(VOTING, **R1_R3), ALL),
    "29 voting robust R1+R3": (with_team_strength(VOTING, **R1_R3), ROB),
    "LogReg R1+R3": (with_team_strength(LOGREG, **R1_R3), ALL),
}
BOOSTING = [name for name in MEMBERS if name != "LogReg R1+R3"]
REFERENCE = "28 voting R1+R3"
LOGREG_WEIGHT = 0.25
NEW_SEEDS = (1, 2, 3)

## 2. Out-of-fold probabilities

Each model is refitted in every fold, encoder included. Results are saved after each model and seed, so an interrupted run
resumes where it stopped. Delete the cache file if a model or the code changes.

In [3]:
CACHE = OUT / "oof_blend.pkl"
oof = pickle.loads(CACHE.read_bytes()) if CACHE.exists() else {}

t0 = time.time()
for seed in (*REPEAT_SEEDS, *NEW_SEEDS):
    for name, (model, columns) in MEMBERS.items():
        if (name, seed) not in oof:
            oof[(name, seed)] = oof_proba(model, X[columns], y, seed=seed)
            CACHE.write_bytes(pickle.dumps(oof))
            print(f"seed {seed:3d} | {name:24s} {fold_accuracy(oof[(name, seed)], y, seed=seed).mean():.4f} ({time.time() - t0:.0f}s)")
print(f"done in {time.time() - t0:.0f}s")

seed  42 | 22 voting                0.6884 (53s)
seed  42 | 23 voting robust         0.6880 (105s)
seed  42 | 24 LightGBM              0.6905 (111s)
seed  42 | 27 LightGBM R1+R3        0.6896 (117s)
seed  42 | 28 voting R1+R3          0.6891 (172s)
seed  42 | 29 voting robust R1+R3   0.6874 (227s)
seed  42 | LogReg R1+R3             0.6865 (228s)
seed   7 | 22 voting                0.6926 (281s)
seed   7 | 23 voting robust         0.6881 (330s)
seed   7 | 24 LightGBM              0.6874 (334s)
seed   7 | 27 LightGBM R1+R3        0.6897 (339s)
seed   7 | 28 voting R1+R3          0.6895 (397s)
seed   7 | 29 voting robust R1+R3   0.6873 (452s)
seed   7 | LogReg R1+R3             0.6816 (453s)
seed 123 | 22 voting                0.6864 (507s)
seed 123 | 23 voting robust         0.6793 (558s)
seed 123 | 24 LightGBM              0.6821 (562s)
seed 123 | 27 LightGBM R1+R3        0.6871 (567s)
seed 123 | 28 voting R1+R3          0.6835 (624s)
seed 123 | 29 voting robust R1+R3   0.6840 (680s)
s

## 3. Scores of the members and of the blends

In [4]:
def boosting_blend(seed):
    return np.mean([oof[(name, seed)] for name in BOOSTING], axis=0)

def logreg_blend(seed):
    return (1 - LOGREG_WEIGHT) * boosting_blend(seed) + LOGREG_WEIGHT * oof[("LogReg R1+R3", seed)]

def scores_of(proba_of, seeds):
    return np.concatenate([fold_accuracy(proba_of(s), y, seed=s) for s in seeds])

candidates = {name: (lambda s, n=name: oof[(n, s)]) for name in MEMBERS}
candidates["boosting blend"] = boosting_blend
candidates["blend + LogReg"] = logreg_blend

results = {}
for label, seeds in (("seeds 42/7/123", REPEAT_SEEDS), ("seeds 1/2/3", NEW_SEEDS)):
    results[label] = {name: scores_of(fn, seeds) for name, fn in candidates.items()}

ref_scores = {label: r[REFERENCE] for label, r in results.items()}
table = pd.DataFrame({
    name: {
        **{f"{label}: acc": r[name].mean() for label, r in results.items()},
        **{f"{label}: vs 28": (r[name] - ref_scores[label]).mean() * 100 for label, r in results.items()},
        **{f"{label}: folds": int((r[name] > ref_scores[label]).sum()) for label, r in results.items()},
    } for name in candidates
}).T
table.to_csv(OUT / "blending.csv")
table.round(4)

,seeds 42/7/123: acc,seeds 1/2/3: acc,seeds 42/7/123: vs 28,seeds 1/2/3: vs 28,seeds 42/7/123: folds,seeds 1/2/3: folds
22 voting,0.6891,0.6844,0.1767,-0.2833,8.0,3.0
23 voting robust,0.6851,0.6832,-0.2233,-0.4033,4.0,5.0
24 LightGBM,0.6867,0.6827,-0.0700,-0.4533,5.0,2.0
27 LightGBM R1+R3,0.6888,0.6863,0.1433,-0.0933,7.0,5.0
28 voting R1+R3,0.6874,0.6873,0.0000,0.0000,0.0,0.0
29 voting robust R1+R3,0.6862,0.6874,-0.1133,0.0167,3.0,7.0
LogReg R1+R3,0.6843,0.6849,-0.3100,-0.2367,4.0,5.0
boosting blend,0.6905,0.6866,0.3133,-0.0633,11.0,6.0
blend + LogReg,0.6907,0.6893,0.3300,0.2033,11.0,10.0


## 4. Do the members make different errors?

Share of battles on which two models predict a different winner, out of fold, on seed 42. The LogReg row is what the proposal
rests on. The two lines below the table compare LightGBM with soft voting on the same 15 folds.

In [5]:
seed = REPEAT_SEEDS[0]
labels = pd.DataFrame({name: oof[(name, seed)] > 0.5 for name in MEMBERS})
disagreement = pd.DataFrame({a: {b: (labels[a] != labels[b]).mean() for b in MEMBERS} for a in MEMBERS})
display(disagreement.round(3))

first = results["seeds 42/7/123"]
for lgbm, vote in (("24 LightGBM", "22 voting"), ("27 LightGBM R1+R3", "28 voting R1+R3")):
    d = first[vote] - first[lgbm]
    print(f"{vote} vs {lgbm}: {d.mean() * 100:+.2f} points, better in {(d > 0).sum()}/15 folds")

,22 voting,23 voting robust,24 LightGBM,27 LightGBM R1+R3,28 voting R1+R3,29 voting robust R1+R3,LogReg R1+R3
22 voting,0.000,0.049,0.039,0.062,0.056,0.059,0.114
23 voting robust,0.049,0.000,0.058,0.070,0.066,0.057,0.118
24 LightGBM,0.039,0.058,0.000,0.055,0.061,0.066,0.115
27 LightGBM R1+R3,0.062,0.070,0.055,0.000,0.034,0.045,0.105
28 voting R1+R3,0.056,0.066,0.061,0.034,0.000,0.034,0.106
29 voting robust R1+R3,0.059,0.057,0.066,0.045,0.034,0.000,0.114
LogReg R1+R3,0.114,0.118,0.115,0.105,0.106,0.114,0.000


22 voting vs 24 LightGBM: +0.25 points, better in 12/15 folds
28 voting R1+R3 vs 27 LightGBM R1+R3: -0.14 points, better in 8/15 folds


## 5. Decision

Rule fixed before running: at least +0.15 points on both sets of seeds, boosting blend against submission 28 and blend + LogReg
against the boosting blend.

In [6]:
def gains(name, against):
    return {label: (r[name] - r[against]).mean() * 100 for label, r in results.items()}

blend_gain = gains("boosting blend", REFERENCE)
logreg_gain = gains("blend + LogReg", "boosting blend")
adopt_blend = all(v >= 0.15 for v in blend_gain.values())
adopt_logreg = all(v >= 0.15 for v in logreg_gain.values())
print("boosting blend vs 28:", {k: round(v, 2) for k, v in blend_gain.items()}, "->", "ADOPT" if adopt_blend else "DO NOT ADOPT")
print("LogReg in the blend: ", {k: round(v, 2) for k, v in logreg_gain.items()}, "->", "ADOPT" if adopt_logreg else "LEAVE OUT")

boosting blend vs 28: {'seeds 42/7/123': np.float64(0.31), 'seeds 1/2/3': np.float64(-0.06)} -> DO NOT ADOPT
LogReg in the blend:  {'seeds 42/7/123': np.float64(0.02), 'seeds 1/2/3': np.float64(0.27)} -> LEAVE OUT


## 6. Summary

| Model | Seeds 42/7/123 | Seeds 1/2/3 | Mean of the 30 folds |
|---|---|---|---|
| 22 voting | 0.6891 | 0.6844 | 0.6868 |
| 23 voting robust | 0.6851 | 0.6832 | 0.6842 |
| 24 LightGBM | 0.6867 | 0.6827 | 0.6847 |
| 27 LightGBM R1+R3 | 0.6888 | 0.6863 | 0.6876 |
| 28 voting R1+R3 | 0.6874 | 0.6873 | 0.6874 |
| 29 voting robust R1+R3 | 0.6862 | 0.6874 | 0.6868 |
| LogReg R1+R3 | 0.6843 | 0.6849 | 0.6846 |
| Boosting blend | 0.6905 | 0.6866 | 0.6886 |
| Blend + LogReg | 0.6907 | 0.6893 | 0.6900 |

- **Decision by the rule fixed before running: neither adopted.** Boosting blend against 28: +0.31 and -0.06 points. LogReg added to
  the blend: +0.02 and +0.27 points.
- **Comparison outside the rule:** blend + LogReg against 28 gives +0.33 (11/15 folds) and +0.20 (10/15), the only result positive on
  both sets of seeds. It was seen after the results, so it needs fresh seeds before it counts.
- **LogReg makes different errors:** it disagrees with the boosting models on 10.5-11.8% of the battles, the boosting models with one
  another on 3.4-7.0%.
- **Soft voting or LightGBM:** with the R1 + R3 encoder they are equal (+0.14 and -0.09 points for LightGBM). With the encoder of
  submissions 22 and 24 soft voting is better (+0.25 and +0.17).
- **Submission 24** has the best public score, but its seed-42 CV (0.6905) was favourable: over 30 folds it is 0.6847, 0.29 points below
  submission 27.
- These 30 folds resample the same 10,000 battles, so they do not make the differences independent measures.